# CW305 MNIST Neural Network — Power Side-Channel Analysis

This notebook captures power traces from a **784→64→64→32→10 fully-connected neural network**
running on the CW305 FPGA and performs side-channel analysis to extract weight information
via **Hamming Weight (HW)** and **Hamming Distance (HD)** leakage models.

**Register map** (from `cw305_user_defines.v`):

| Symbol | Address | Direction | Description |
|---|---|---|---|
| Image pixels | 0x0000–0x030F (0–783) | W | One byte per pixel; each pixel needs its own `fpga_write(i, [px])` call |
| `REG_NN_GO` | 0x1000 | W | Write 1 to start inference; hardware auto-clears |
| `REG_NN_RESULT` | 0x1001 | R | Argmax output (0–9) |
| `REG_NN_DONE` | 0x1002 | R | High for **1 crypt_clk cycle** when done (easy to miss over USB) |
| `REG_USER_LED` | 0x1003 | R/W | LED control (bits [1:0]) |
| `REG_NN_STATE` | 0x1004 | R | Current FSM state (debug) |

**FSM states:** IDLE=0, LAYER1_MM=1, LAYER1_RELU=2, LAYER2_MM=3, LAYER2_RELU=4,
LAYER3_MM=5, LAYER3_RELU=6, LAYER4_MM=7, ARGMAX=8, DONE=9.

**Important — image write addressing:**  
`fpga_write(addr, data)` shifts `addr` left by `bytecount_size=7` before sending to USB.  
A single call `fpga_write(0, [784 bytes])` writes ALL bytes to `reg_address=0` (overwriting
each other), because the bytecnt field (lower 7 bits) does not increment `reg_address`.  
→ **Always write each pixel with its own call: `fpga_write(i, [pixel])`**.

**NN forward-pass cycle budget (at 10 MHz, sequential matrix multiply):**

| Layer | Dimensions | Approx cycles | Time (10 MHz) |
|---|---|---|---|
| Layer 1 MM | 784→64 | 64×(784+3) = 50,368 | 5.04 ms |
| Layer 1 ReLU | 64 | 128 | 0.01 ms |
| Layer 2 MM | 64→64 | 64×(64+3) = 4,288 | 0.43 ms |
| Layer 2 ReLU | 64 | 128 | 0.01 ms |
| Layer 3 MM | 64→32 | 32×(64+3) = 2,144 | 0.21 ms |
| Layer 3 ReLU | 32 | 64 | 0.01 ms |
| Layer 4 MM | 32→10 | 10×(32+3) = 350 | 0.04 ms |
| Argmax | 10 | ~15 | <0.01 ms |
| **Total** | | **~57,485 cycles** | **~5.75 ms** |

At 10 MHz × 4× ADC oversampling = **40 MS/s**, the CW-Lite's 24,400-sample buffer covers
only **~6,100 FPGA cycles (0.61 ms) = ~10.7% of the forward pass**.  
→ Use `scope.adc.offset` to slide the capture window and target specific layers.

## 1 — Imports and constants

In [ ]:
import chipwhisperer as cw
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
from tqdm.notebook import tnrange, tqdm

BITFILE      = 'test2/test2.runs/impl_1/cw305_top.bit'
DEFINES_FILE = 'test2.srcs/sources_1/new/cw305_user_defines.v'
DATASET_FILE = 'MNIST-10000-784.csv'
TARGET_FREQ  = 10e6   # FPGA crypt_clk in Hz; safe starting point, range 630 kHz–167 MHz

# Derived constants
SAMPLING_RATE = TARGET_FREQ * 4   # extclk_x4: ADC runs at 4× the FPGA clock

# Approximate cycle counts per computation stage (matrix_multiply pipeline depth = 3)
LAYER_CYCLES = {
    'L1-MatMul':  64 * (784 + 3),   # 784→64
    'L1-ReLU':    64 * 2,
    'L2-MatMul':  64 * (64 + 3),    # 64→64
    'L2-ReLU':    64 * 2,
    'L3-MatMul':  32 * (64 + 3),    # 64→32
    'L3-ReLU':    32 * 2,
    'L4-MatMul':  10 * (32 + 3),    # 32→10
    'Argmax':     15,
}
TOTAL_CYCLES = sum(LAYER_CYCLES.values())

# Cumulative start cycle for each stage (used to annotate plots)
cum = 0
LAYER_START_CYCLES = {}
for name, cyc in LAYER_CYCLES.items():
    LAYER_START_CYCLES[name] = cum
    cum += cyc

print(f'ChipWhisperer version : {cw.__version__}')
print(f'NN total cycles       ≈ {TOTAL_CYCLES:,}  ({TOTAL_CYCLES/TARGET_FREQ*1000:.2f} ms at {TARGET_FREQ/1e6:.0f} MHz)')
print(f'ADC sampling rate     : {SAMPLING_RATE/1e6:.0f} MS/s')
print(f'Layer start cycles    :', {k: v for k, v in LAYER_START_CYCLES.items()})

## 2 — Connect to scope

`cw.scope()` auto-detects a connected CW-Lite or CW-Pro over USB.  
`default_setup()` applies sensible gain/clock defaults (we override clock and trigger below).  
The `try/except dis()` blocks ensure any previous session is cleanly released before reconnecting.

In [ ]:
try:
    scope.dis()
except Exception:
    pass
try:
    target.dis()
except Exception:
    pass

scope = cw.scope()
scope.default_setup()
print(scope)

## 3 — Subclass CW305 and program the FPGA

Two class attributes must match the Verilog design:
- `registers = 5` — number of `` `define REG_NN_* `` lines in `cw305_user_defines.v`  
  (image pixel addresses 0–783 are **not** register defines and do not count here)
- `bytecount_size = 7` — must equal `pBYTECNT_SIZE` in `cw305_top.v`

`cw.target()` programs the FPGA via JTAG and calls `slurp_defines()` to populate
`target.REG_NN_GO`, `target.REG_NN_RESULT`, etc. from the defines file.

In [ ]:
class MyCW305(cw.targets.CW305):
    """CW305 subclass for the 5-register MNIST NN design."""
    def __init__(self):
        super().__init__()
        self.registers      = 5   # REG_NN_GO, REG_NN_RESULT, REG_NN_DONE, REG_USER_LED, REG_NN_STATE
        self.bytecount_size = 7   # pBYTECNT_SIZE in cw305_top.v

try:
    target.dis()
except Exception:
    pass

target = cw.target(scope, MyCW305,
                   bsfile        = BITFILE,
                   force         = True,
                   fpga_id       = '100t',
                   platform      = 'cw305',
                   defines_files = [DEFINES_FILE])

print('FPGA programmed.  Register addresses from defines file:')
for name in ['REG_NN_GO', 'REG_NN_RESULT', 'REG_NN_DONE', 'REG_USER_LED', 'REG_NN_STATE']:
    val = getattr(target, name, 'NOT FOUND')
    print(f'  target.{name} = 0x{val:04X}' if isinstance(val, int) else f'  target.{name} = {val}')

## 4 — Configure the CW305 onboard PLL

The CW305 has a programmable CDCE906 PLL. Channel 1 drives FPGA pin N13 (`pll_clk1`),
which feeds `BUFG → crypt_clk` and is forwarded back out via `ODDR → tio_clkout (M16)`
so the scope can lock its ADC to the same clock (see Cell 5).

In [ ]:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)   # channel 0 off
target.pll.pll_outenable_set(True,  1)   # channel 1 → pll_clk1 (N13) → crypt_clk
target.pll.pll_outenable_set(False, 2)   # channel 2 off
target.pll.pll_outfreq_set(TARGET_FREQ, 1)
time.sleep(0.1)
print(f'PLL channel 1 set to {TARGET_FREQ/1e6:.1f} MHz')

## 5 — Lock scope ADC to FPGA clock

**Signal path:** `pll_clk1 (N13) → BUFG → crypt_clk → ODDR → tio_clkout (M16) → 20-pin → HS1`

- `hs2` is output-only — set to `'disabled'` so it does not fight HS1.
- `extclk_x4` locks the ADC to 4× the incoming clock on HS1 (40 MHz at 10 MHz input).
- Falls back to `clkgen_x4` if HS1 has no clock (20-pin ribbon not connected).

> **Physical requirement:** the 20-pin ribbon cable must be seated on both boards
> before running this cell — it carries both `tio_clkout → HS1` (clock) and
> `tio_trigger → tio4` (trigger).

In [ ]:
scope.io.hs2 = 'disabled'           # HS2 is output-only; disable so it doesn't drive HS1
scope.clock.adc_src = 'extclk_x4'  # lock to FPGA clock arriving on HS1 via tio_clkout
scope.clock.reset_adc()
time.sleep(0.2)

if scope.clock.adc_locked and scope.clock.adc_freq > 1e6:
    print(f'ADC locked to EXTCLK (HS1).  Freq = {scope.clock.adc_freq/1e6:.2f} MHz')
    print('Good — ADC is synchronised to the FPGA clock (no jitter).')
else:
    print('WARNING: extclk (HS1) not detected — falling back to clkgen_x4.')
    print('Check: 20-pin ribbon cable seated on both boards.')
    scope.clock.adc_src = 'clkgen_x4'
    scope.clock.clkgen_freq = TARGET_FREQ
    scope.clock.reset_adc()
    time.sleep(0.1)
    assert scope.clock.adc_locked, 'ADC lock failed — check USB and board power.'
    print(f'ADC locked to CLKGEN.  Freq = {scope.clock.adc_freq/1e6:.2f} MHz')
    print('WARNING: ADC is NOT synchronised to FPGA — traces will have phase jitter.')

## 6 — Configure trigger and ADC capture window

**Trigger path:** `REG_NN_GO write (USB) → go_stretch_active (64 USB clocks)
→ CDC sync to crypt_clk → nn_start_pulse (rising-edge detect)
→ trig_stretch counter (64 crypt_clk cycles, ~6.4 µs) → tio_trigger (M16) → tio4`

- `tio4` defaults to `'high_z'` which IS the input/receive state.  
  **`'gpio_in'` is NOT a valid mode** — valid options: `high_z`, `serial_tx`, `gpio_low`, `gpio_high`.
- `scope.adc.offset`: number of samples to skip after the trigger before recording.
  Use this to slide the capture window deeper into the NN computation.

**Capture window vs. full forward pass:**  
CW-Lite maximum = 24,400 samples = 6,100 FPGA cycles = 0.61 ms at 10 MHz.  
Full NN ≈ 57,485 cycles = 5.75 ms → ~10.7% coverage per window.  
To cover Layer 1 alone (~50,368 cycles), you need ~9 non-overlapping windows.

In [ ]:
# tio4 is already 'high_z' (input) by default — no direction change needed
scope.trigger.triggers = 'tio4'
scope.adc.basic_mode   = 'rising_edge'
scope.adc.offset       = 0       # start recording immediately at trigger; increase to shift window
scope.adc.samples      = 24400   # CW-Lite max; lower this if transfers are too slow

MAX_SAMPLES    = scope.adc.samples
WINDOW_CYCLES  = MAX_SAMPLES / 4   # FPGA cycles visible per capture window
WINDOW_MS      = MAX_SAMPLES / SAMPLING_RATE * 1000

print(f'tio4 state     : {scope.io.tio4}')          # should print high_z
print(f'Trigger source : {scope.trigger.triggers}')
print(f'ADC source     : {scope.clock.adc_src}')
print(f'Samples        : {scope.adc.samples}')
print(f'Capture window : {WINDOW_MS:.2f} ms = {WINDOW_CYCLES:.0f} FPGA cycles')
print(f'Full NN needs  : {TOTAL_CYCLES/TARGET_FREQ*1000:.2f} ms = {TOTAL_CYCLES:,} FPGA cycles')
print(f'Window coverage: {WINDOW_CYCLES/TOTAL_CYCLES*100:.1f}% of total forward pass')

## 7 — Load MNIST dataset

The CSV has one row per image: column `label` (0–9) plus pixel columns `1x1`…`28x28`.
Images are binarised at threshold 127 because the FPGA image_memory stores each pixel
as a 32-bit signed integer (0x00000000 or 0x00000001), and the weights are trained on
binary inputs.

In [ ]:
df     = pd.read_csv(DATASET_FILE)
labels = df['label'].values
pixels = df.drop('label', axis=1).values   # shape: (N, 784), dtype uint8 or int

def get_binary_image(index):
    """Return binarised image (np.uint8, 784 values 0/1) and its integer label."""
    return (pixels[index] > 127).astype(np.uint8), int(labels[index])

img0, lbl0 = get_binary_image(0)
print(f'Dataset    : {len(labels)} images, classes {sorted(np.unique(labels).tolist())}')
print(f'Image 0    : label={lbl0}, non-zero pixels={img0.sum()}/784')

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(img0.reshape(28, 28), cmap='gray')
ax.set_title(f'Sample 0 — Label: {lbl0}')
ax.axis('off')
plt.show()

## 8 — Image write and inference helpers

### Why individual pixel writes are required

`fpga_write(addr, data)` internally computes `usb_address = addr << bytecount_size` (here `<< 7`).
When called with a list of 784 bytes at `addr=0`, the USB address stays at 0 for all bytes —
`reg_address = usb_address >> 7 = 0` throughout, so all 784 bytes hit `image_memory[0]`
(last byte wins) and `image_memory[1..783]` remain zero.  
The fix is **one `fpga_write(i, [pixel])` per pixel**, which produces
`usb_address = i << 7` → `reg_address = i` → correct `image_memory[i]` write.

### Why to poll REG_NN_STATE instead of REG_NN_DONE

`REG_NN_DONE` is derived from `(current_state == DONE)` which is high for exactly
**one crypt_clk cycle** before transitioning back to IDLE.  At 10 MHz that is 100 ns —
far shorter than a USB round-trip.  `REG_NN_STATE` holds the 4-bit FSM state and is
readable while the computation is running; waiting for it to return to IDLE (0) is reliable.

In [ ]:
def write_image(img_bin):
    """Write 784 binary pixels to FPGA image_memory (one fpga_write per pixel)."""
    for i, px in enumerate(img_bin):
        target.fpga_write(i, [int(px)])


def wait_for_nn_done(timeout_s=1.0):
    """Poll REG_NN_STATE until inference finishes; return True on success, False on timeout.

    Strategy: wait for state to leave IDLE (inference started), then wait for it to
    return to IDLE (inference finished and DONE→IDLE transition occurred).
    Falling through without seeing a non-IDLE state is also a success guard — the NN
    may have finished before the first poll if it ran very fast at high clock frequencies.
    """
    deadline = time.time() + timeout_s
    # Phase 1: wait for state to leave IDLE
    saw_active = False
    while time.time() < deadline:
        state = target.fpga_read(target.REG_NN_STATE, 1)[0]
        if state != 0:
            saw_active = True
            break
        time.sleep(0.0001)
    if not saw_active:
        # NN may have already finished (very fast clock) — assume done
        return True
    # Phase 2: wait for state to return to IDLE
    while time.time() < deadline:
        state = target.fpga_read(target.REG_NN_STATE, 1)[0]
        if state == 0:
            return True
        time.sleep(0.0001)
    return False  # timeout


def classify_and_capture(img_bin, verbose=False):
    """Write image, arm scope, trigger NN, capture power trace, return (prediction, trace).

    Returns (None, None) on capture timeout (trigger not received within scope.adc.timeout).
    Always waits for the NN to finish before reading REG_NN_RESULT so the result register
    is valid (not a leftover from the previous inference).
    """
    write_image(img_bin)

    scope.arm()  # arm BEFORE triggering so the rising edge is not missed

    # A single write to REG_NN_GO is sufficient; the FPGA's go_stretch logic holds
    # go_stretch_active high for 64 USB clocks (~1.3 µs) to safely cross the CDC
    # boundary into crypt_clk and generate nn_start_pulse.
    target.fpga_write(target.REG_NN_GO, [0x01])

    timed_out = scope.capture()   # blocks until sample buffer full or timeout
    if timed_out:
        if verbose:
            print('WARNING: scope.capture() timed out — trigger not received.')
            print('  Check 20-pin cable, tio4 wiring, and FPGA trigger logic.')
        return None, None

    trace = scope.get_last_trace()

    # scope.capture() returns as soon as the sample buffer is full, which is much
    # earlier than the end of the NN computation.  Wait for the NN to finish.
    finished = wait_for_nn_done(timeout_s=1.0)
    if not finished and verbose:
        print('WARNING: wait_for_nn_done() timed out — REG_NN_RESULT may be stale.')

    result = target.fpga_read(target.REG_NN_RESULT, 1)[0]
    return result, trace


# --- Quick functional test ---
print('Testing classify_and_capture on image 0...')
img0, lbl0 = get_binary_image(0)
res0, trace0 = classify_and_capture(img0, verbose=True)
print(f'Image 0 — actual: {lbl0}, predicted: {res0}')
if trace0 is not None:
    print(f'Trace  : {len(trace0)} samples, range [{trace0.min():.4f}, {trace0.max():.4f}]')

## 9 — Leakage model functions

### Hamming Weight (HW)
Power consumption of CMOS circuits is dominated by switching activity, which correlates
with the number of 1-bits in the data being processed.  For the matrix-multiply pipeline:

- At step `p` for output neuron `j`:  `product = input[p] × weight1[j][p]`
- Since `input[p] ∈ {0, 1}`:  `HW(product) = input[p] × HW(weight1[j][p])`
- Correlating `input[p]` with the trace at the time step where pixel `p` is fetched
  reveals `HW(weight1[j][p])`.

### Hamming Distance (HD)
Leakage also occurs at register transitions.  `HD(old, new) = HW(old XOR new)`
models the power cost of flipping bits in a register.

In [ ]:
def hamming_weight(x):
    """Bit count of integer or integer array."""
    x = np.asarray(x, dtype=np.int64).ravel()
    return np.array([bin(int(v) & 0xFFFFFFFF).count('1') for v in x])


def hamming_distance(a, b):
    """Bit-flip count between two integers or arrays of integers."""
    return hamming_weight(np.asarray(a, dtype=np.int64) ^ np.asarray(b, dtype=np.int64))


def pearson_corr_traces(traces, model_values):
    """Pearson correlation coefficient between each sample column and a scalar label vector.

    Parameters
    ----------
    traces       : ndarray (N, T) — N power traces each with T samples
    model_values : ndarray (N,)   — HW or HD prediction for each trace

    Returns
    -------
    corr : ndarray (T,) — Pearson r at each sample position
    """
    T = np.asarray(traces, dtype=float)
    L = np.asarray(model_values, dtype=float)
    T -= T.mean(axis=0)
    L -= L.mean()
    std_T = np.std(traces, axis=0)
    std_L = np.std(model_values)
    denom = std_T * std_L * len(model_values)
    denom = np.where(denom == 0, np.nan, denom)
    return (T * L[:, None]).sum(axis=0) / denom


# Sanity checks
print('hamming_weight([0, 1, 3, 255, 0xAA]) =', hamming_weight([0, 1, 3, 255, 0xAA]))
print('hamming_distance(0xFF, 0x0F)         =', hamming_distance(0xFF, 0x0F))   # 4 bit flips

## 10 — Single-image capture and interactive trace viewer

The Plotly figure below is **fully interactive**:
- **Zoom/pan**: drag on the trace area, or scroll to zoom in/out
- **Range slider**: the bar at the bottom lets you zoom to any region without losing the overview
- **Hover**: shows time in µs and approximate FPGA cycle count simultaneously
- **Layer annotations**: dashed vertical lines mark the expected start of each computation stage
  (only those within the current capture window are shown)

Change `SAMPLE_IDX` and re-run to inspect a different image.

In [ ]:
SAMPLE_IDX = 0   # change to any index in the dataset

img, lbl = get_binary_image(SAMPLE_IDX)
res, trace = classify_and_capture(img, verbose=True)
print(f'Image {SAMPLE_IDX} — actual: {lbl}, predicted: {res}')

if trace is not None:
    n = len(trace)
    t_us        = np.arange(n) / SAMPLING_RATE * 1e6   # µs axis
    fpga_cycles = np.arange(n) / 4                      # FPGA cycle axis (at 4× oversampling)
    capture_offset_cycles = scope.adc.offset / 4

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=t_us, y=trace,
        name='Power trace',
        line=dict(color='steelblue', width=0.8),
        hovertemplate=(
            'Time: %{x:.2f} µs<br>'
            'FPGA cycle ≈ ' + f'{capture_offset_cycles:.0f}' + '+%{customdata:.0f}<br>'
            'ADC: %{y:.5f}<extra></extra>'
        ),
        customdata=fpga_cycles,
    ))

    # Trigger marker at t=0
    fig.add_vline(x=0, line=dict(color='red', dash='dash', width=1.5),
                  annotation_text='▲ Trigger', annotation_position='top left',
                  annotation_font=dict(color='red', size=11))

    # Layer boundary annotations (only those within the capture window)
    layer_colors = ['darkorange', 'green', 'purple', 'brown',
                    'teal', 'darkgreen', 'darkred', 'navy']
    window_end_cycles = capture_offset_cycles + n / 4
    for (layer_name, abs_start_cyc), color in zip(LAYER_START_CYCLES.items(), layer_colors):
        rel_cyc = abs_start_cyc - capture_offset_cycles   # cycles relative to window start
        if 0 <= rel_cyc <= n / 4:
            t_line = rel_cyc / (TARGET_FREQ / 1e6)  # µs
            fig.add_vline(
                x=t_line,
                line=dict(color=color, dash='dot', width=1.2),
                annotation_text=layer_name,
                annotation_textangle=-90,
                annotation_position='top right',
                annotation_font=dict(color=color, size=10),
            )

    fig.update_layout(
        title=dict(
            text=(f'Power Trace — Image {SAMPLE_IDX}  |  Actual: {lbl}, Predicted: {res}'
                  f'  |  {TARGET_FREQ/1e6:.0f} MHz  |  offset={scope.adc.offset} samples'),
            font=dict(size=13),
        ),
        xaxis=dict(
            title='Time (µs)',
            rangeslider=dict(visible=True, thickness=0.07),
        ),
        yaxis=dict(title='ADC Value (normalised)'),
        xaxis2=dict(
            title='FPGA cycles (approx, from trigger)',
            overlaying='x', side='top',
            tickmode='array',
            tickvals=np.linspace(0, t_us[-1], 9),
            ticktext=[f'{int(capture_offset_cycles + c):.0f}'
                      for c in np.linspace(0, n / 4, 9)],
        ),
        hovermode='x unified',
        height=520,
        margin=dict(t=70, b=90),
        legend=dict(x=0.01, y=0.99),
    )
    fig.show()

    print(f'Window: offset={scope.adc.offset}  →  '
          f'FPGA cycles {capture_offset_cycles:.0f}–{window_end_cycles:.0f}  '
          f'({(window_end_cycles-capture_offset_cycles)/TOTAL_CYCLES*100:.1f}% of total forward pass)')

## 11 — Multi-window capture (offset sweep)

Because one capture window covers only ~10% of the forward pass, sliding `scope.adc.offset`
lets us reconstruct the full trace by stitching multiple windows of the same image.

**Important:** the image must be re-written and the NN re-triggered for each window
because the FPGA state resets between runs.  The NN is deterministic for a fixed input,
so the traces should be nearly identical across repeated runs of the same image.

Adjust `N_WINDOWS` and `WINDOW_STEP` to control coverage vs. measurement time.

In [ ]:
N_WINDOWS   = 4                        # number of adjacent windows to capture
WINDOW_STEP = MAX_SAMPLES              # advance by one full window each time (no overlap)
# Set WINDOW_STEP < MAX_SAMPLES for overlap (useful for smoothing at window boundaries)

offsets = [i * WINDOW_STEP for i in range(N_WINDOWS)]
print(f'Capturing {N_WINDOWS} windows: offsets {offsets}')
print(f'Covering FPGA cycles {offsets[0]//4} – {(offsets[-1] + MAX_SAMPLES)//4}'
      f'  ({(N_WINDOWS * MAX_SAMPLES / 4) / TOTAL_CYCLES * 100:.1f}% of forward pass)')

window_traces = []
img0, lbl0 = get_binary_image(0)

for off in tqdm(offsets, desc='Capturing windows'):
    scope.adc.offset = off
    _res, _tr = classify_and_capture(img0)
    window_traces.append(_tr)

scope.adc.offset = 0   # restore default for subsequent cells

# --- Stitched interactive plot ---
fig = go.Figure()
palette = ['steelblue', 'darkorange', 'green', 'crimson',
           'purple', 'brown', 'teal', 'navy']

for i, (off, tr) in enumerate(zip(offsets, window_traces)):
    if tr is None:
        continue
    t_abs_us = (np.arange(len(tr)) + off) / SAMPLING_RATE * 1e6
    fig.add_trace(go.Scatter(
        x=t_abs_us, y=tr,
        name=f'Window {i} (offset={off})',
        line=dict(color=palette[i % len(palette)], width=0.7),
        opacity=0.85,
    ))

# Layer boundaries
for (name, start_cyc), color in zip(LAYER_START_CYCLES.items(), palette):
    t_us_line = start_cyc / TARGET_FREQ * 1e6
    fig.add_vline(x=t_us_line,
                  line=dict(color='grey', dash='dot', width=0.8),
                  annotation_text=name, annotation_textangle=-90,
                  annotation_font=dict(size=9, color='grey'),
                  annotation_position='top right')

fig.update_layout(
    title=f'Multi-window power trace — Image 0, Label {lbl0}',
    xaxis=dict(title='Time (µs)', rangeslider=dict(visible=True, thickness=0.07)),
    yaxis=dict(title='ADC Value'),
    hovermode='x unified',
    height=480,
    margin=dict(b=90),
)
fig.show()

## 12 — Batch capture

Captures `N` traces with `scope.adc.offset = 0` (start of inference).  
For SCA, capture at least **100–1000 traces** for reliable Pearson correlation estimates.
To target a specific layer, change `scope.adc.offset` before running this cell.

In [ ]:
N = 20   # number of traces to collect; increase to 100–1000 for real SCA

scope.adc.offset = 0   # window start (change to target a specific layer)

traces_batch  = []   # list of 1-D trace arrays
images_batch  = []   # list of 784-pixel binary images (for leakage model)
labels_batch  = []   # ground-truth labels
preds_batch   = []   # FPGA predictions
n_timeout     = 0

for i in tnrange(N, desc='Capturing'):
    img, lbl = get_binary_image(i)
    res, trace = classify_and_capture(img)
    if trace is None:
        n_timeout += 1
        continue
    traces_batch.append(trace)
    images_batch.append(img.copy())
    labels_batch.append(lbl)
    preds_batch.append(res)

traces_batch = np.array(traces_batch, dtype=np.float32)
images_batch = np.array(images_batch, dtype=np.uint8)

correct = sum(l == r for l, r in zip(labels_batch, preds_batch))
total   = len(labels_batch)

print(f'Captured  : {total}/{N} traces  ({n_timeout} timeouts)')
print(f'Accuracy  : {correct}/{total} = {correct/total*100:.1f}%')
print(f'Trace shape: {traces_batch.shape}')
print('Results   :', list(zip(labels_batch, preds_batch)))

## 13 — Side-channel analysis: HW correlation

**Goal:** find which time samples correlate with each pixel value, revealing *when* each
pixel is processed and (via the correlation sign and magnitude) the Hamming Weight of
the corresponding Layer-1 weight.

**Model:** at time step `t` corresponding to pixel `p` in neuron `j`,
the ADC value is approximately:

```
trace[t] ≈ a × HW(input[p] × weight1[j][p])  +  noise
         = a × input[p] × HW(weight1[j][p])   +  noise
```

So `Pearson(trace[:, t], input[:, p])` peaks at the sample where pixel `p` is multiplied
into neuron `j`, and the peak height is proportional to `HW(weight1[j][p])`.

> Collect ≥100 traces for statistically significant correlations.

In [ ]:
if len(traces_batch) < 5:
    print(f'Only {len(traces_batch)} traces available — collect more in Cell 12 for reliable SCA.')
else:
    # Vectorised Pearson: corr[p, t] = Pearson(images_batch[:, p], traces_batch[:, t])
    X = images_batch.astype(np.float64)       # (N, 784)
    Y = traces_batch.astype(np.float64)       # (N, T)

    Xc = X - X.mean(axis=0)                  # zero-mean per pixel
    Yc = Y - Y.mean(axis=0)                  # zero-mean per sample

    std_X = X.std(axis=0)                     # (784,)  — constant pixels → std=0 → NaN
    std_Y = Y.std(axis=0)                     # (T,)
    N_tr  = len(traces_batch)

    # corr[p, t] = (Xc[:, p] · Yc[:, t]) / (N * std_X[p] * std_Y[t])
    denom = np.outer(std_X, std_Y) * N_tr     # (784, T)
    denom = np.where(denom == 0, np.nan, denom)
    corr  = (Xc.T @ Yc) / denom              # (784, T)

    # Aggregate: max |r| across all 784 pixels at each time sample
    max_abs_corr = np.nanmax(np.abs(corr), axis=0)   # (T,)

    t_us = np.arange(traces_batch.shape[1]) / SAMPLING_RATE * 1e6
    mean_trace = traces_batch.mean(axis=0)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=['Mean power trace', 'Max |Pearson r| across all 784 pixels'],
    )

    fig.add_trace(
        go.Scatter(x=t_us, y=mean_trace, name='Mean trace',
                   line=dict(color='steelblue', width=0.8)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=t_us, y=max_abs_corr, name='Max |r|',
                   line=dict(color='crimson', width=0.9),
                   fill='tozeroy', fillcolor='rgba(220,50,50,0.15)'),
        row=2, col=1
    )

    fig.update_xaxes(title_text='Time (µs)', row=2, col=1,
                     rangeslider=dict(visible=True, thickness=0.07))
    fig.update_yaxes(title_text='ADC Value', row=1, col=1)
    fig.update_yaxes(title_text='|Pearson r|', row=2, col=1)
    fig.update_layout(
        title=f'HW Leakage — {N_tr} traces, offset={scope.adc.offset}',
        height=600, hovermode='x unified',
    )
    fig.show()

    # Report strongest single (pixel, sample) correlation
    peak_idx = np.unravel_index(np.nanargmax(np.abs(corr)), corr.shape)
    peak_pix, peak_smp = peak_idx
    peak_r   = corr[peak_idx]
    peak_us  = peak_smp / SAMPLING_RATE * 1e6
    peak_cyc = peak_smp / 4 + scope.adc.offset / 4
    # Estimate which neuron and which k-step this cycle falls into (Layer 1)
    l1_neuron  = int(peak_cyc // (784 + 3))
    l1_k_step  = int(peak_cyc % (784 + 3))
    print(f'Strongest correlation: pixel={peak_pix}, sample={peak_smp}'
          f'  t={peak_us:.2f} µs, FPGA cycle≈{peak_cyc:.0f}, r={peak_r:.4f}')
    print(f'  → Layer-1 estimate: neuron≈{l1_neuron}, k-step≈{l1_k_step}'
          f'  (pixel {l1_k_step} of neuron {l1_neuron})')

## 14 — Correlation heatmap (pixel × sample)

Bright red or blue bands (positive / negative correlation) reveal **when** each pixel
is accessed by the FPGA hardware.  The slope of a band across pixels indicates the
sequential order in which pixels are multiplied.  Gaps between bands correspond to
the 3-cycle pipeline latency at the start of each new output neuron.

The heatmap is subsampled for display; the full `corr` array is still available for
offline analysis.

In [ ]:
if len(traces_batch) >= 5 and 'corr' in dir():
    # Subsample for display: at most 200 pixels and 800 samples
    P, T_ = corr.shape
    sp = max(1, P  // 200)
    st = max(1, T_ // 800)

    sub_corr = corr[::sp, ::st]
    sub_t    = t_us[::st]
    sub_pix  = np.arange(P)[::sp]

    fig = go.Figure(data=go.Heatmap(
        z=sub_corr,
        x=sub_t,
        y=sub_pix,
        colorscale='RdBu',
        zmid=0,
        zmin=-0.5, zmax=0.5,
        colorbar=dict(title='Pearson r'),
        hoverongaps=False,
        hovertemplate='Time: %{x:.2f} µs<br>Pixel: %{y}<br>r: %{z:.4f}<extra></extra>',
    ))

    fig.update_layout(
        title='HW Leakage Map — r(pixel_p, trace_sample_t)',
        xaxis_title='Time (µs)',
        yaxis_title='Pixel index (0–783)',
        height=520,
    )
    fig.show()
    print('Red/blue bands = strong correlation at that (pixel, time) pair.')
    print('The pattern of bands reveals the sequential access order of the matrix multiply.')
else:
    print('Run Cell 13 first to compute the correlation matrix.')

## 15 — Save traces to disk

Saves traces, images, labels, and metadata as a compressed NumPy archive
for offline analysis or re-loading in a fresh session.

In [ ]:
import os

save_path = f'traces_mnist_nn_offset{scope.adc.offset}.npz'

np.savez_compressed(
    save_path,
    traces   = traces_batch,
    images   = images_batch,
    labels   = np.array(labels_batch),
    preds    = np.array(preds_batch),
    metadata = np.array([TARGET_FREQ, SAMPLING_RATE, scope.adc.samples, scope.adc.offset]),
)

size_kb = os.path.getsize(save_path) / 1024
print(f'Saved  : {save_path}  ({size_kb:.1f} KB)')
print(f'traces : {traces_batch.shape}  (N × T  float32)')
print(f'images : {images_batch.shape}  (N × 784 uint8)')
print(f'metadata: [TARGET_FREQ, SAMPLING_RATE, samples, offset]'
      f' = {[TARGET_FREQ, SAMPLING_RATE, scope.adc.samples, scope.adc.offset]}')

# To reload in a new session:
#   d = np.load(save_path)
#   traces = d['traces'];  images = d['images'];  labels = d['labels']

## 16 — Disconnect

Always disconnect cleanly to release the USB device.  
To reconnect, re-run Cells 2–6.

In [ ]:
try:
    scope.dis()
    print('Scope disconnected.')
except Exception as e:
    print(f'Scope dis() error: {e}')
try:
    target.dis()
    print('Target disconnected.')
except Exception as e:
    print(f'Target dis() error: {e}')